In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from lime.lime_text import LimeTextExplainer
from functools import partial
import numpy as np
import shap
import matplotlib.pyplot as plt
# New imports for Integrated Gradients (using Captum)
from captum.attr import IntegratedGradients
from captum.attr import TokenReferenceBase
# Removed: from captum.attr.visualization import visualize_text (due to ModuleNotFoundError)
# Added: Robust custom visualization imports
from IPython.display import HTML, display

import numpy as np
import warnings
warnings.filterwarnings('ignore') # Suppress Captum and Transformer warnings for cleaner output

# --- Custom Visualization Function (Replaces Captum's visualize_text) ---
def plot_text_heatmap(tokens, attributions):
    """
    Generates an HTML representation of the text with attribution heatmap.
    Positive scores (driving prediction) are green, negative scores are red.
    """
    if not tokens or not attributions:
        return HTML("<p>No tokens or attributions to display.</p>")

    # Normalize attributions for color intensity (scale to -1 to 1)
    max_abs = np.max(np.abs(attributions))

    # Handle case where all scores are near zero
    if max_abs == 0:
        norm_attributions = np.zeros_like(attributions)
    else:
        norm_attributions = attributions / max_abs

    html_output = '<p><strong>Integrated Gradients Heatmap:</strong></p>'
    html_output += '<div style="line-height: 1.8; font-size: 14px; padding: 10px; border: 1px solid #ddd; border-radius: 5px; background-color: #f9f9f9;">'

    for token, score in zip(tokens, norm_attributions):
        # Determine color and intensity
        intensity = min(1.0, abs(score) * 2.5) # Amplify intensity slightly for better visibility

        if score > 0:
            # Green for positive contribution (to the predicted class)
            style = f'background-color: rgba(0, 128, 0, {intensity:.2f}); color: white; padding: 1px 2px; border-radius: 3px; margin: 1px;'
        elif score < 0:
            # Red for negative contribution (against the predicted class)
            style = f'background-color: rgba(255, 0, 0, {intensity:.2f}); color: white; padding: 1px 2px; border-radius: 3px; margin: 1px;'
        else:
            # Neutral/Zero contribution
            style = 'background-color: transparent; color: #333; padding: 1px 2px; margin: 1px;'

        # Replace subword tokenization prefix (like '##') for cleaner display
        display_token = token.replace('##', '')

        html_output += f'<span style="{style}">{display_token}</span> '

    html_output += '</div>'
    return HTML(html_output)
# --- End of Custom Visualization Function ---


# --- 1. Configuration and Model Loading (Mocked for environment setup) ---
# NOTE: In a live Colab, you would replace these lines with your actual saved model path.
MODEL_NAME = 'google/electra-small-discriminator'
NUM_LABELS = 3
CLASS_NAMES = ['HAM', 'SPAM', 'AI-SPAM'] # Ensure this matches your final class mapping

print("--- Initializing Model and Tokenizer ---")
try:
    # Load the fine-tuned model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    # Assuming your fine-tuned model is loaded from a path, we mock loading the base model here
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
    # Set model to evaluation mode
    model.eval()
    print("Model and Tokenizer loaded successfully.")
except Exception as e:
    print(f"Error loading model: {e}. Ensure you have saved your fine-tuned model weights correctly.")
    # Exit or handle error gracefully in a real script

# --- 2. Define the Prediction Function for LIME/SHAP (Kept for LIME) ---
def predictor(texts):
    """Tokenizes a list of texts and returns prediction probabilities."""
    with torch.no_grad():
        inputs = tokenizer(texts,
                           return_tensors="pt",
                           padding=True,
                           truncation=True,
                           max_length=128)
        outputs = model(**inputs)
        probabilities = F.softmax(outputs.logits, dim=1).cpu().numpy()
    return probabilities

# --- 3. Example Data for Explanation ---
example_ai_spam_text = (
    "URGENT NOTICE: Your Retail Rewards account balance has been suspended due to "
    "an unauthorized purchase attempt on 25/09/2025. Please confirm your identity "
    "and update your details immediately via the dedicated Retail Secure Link. "
    "Failure to act within 2 hours will result in permanent loss of all accumulated "
    "Loyalty Points and account closure. Click here now: [URL]"
)
example_label_index = 2 # Target class for explanation (AI-SPAM)

# --- 4. LIME Implementation (Local Interpretability) Analysis ---
print("\n--- 4. LIME (Local Interpretability) Analysis ---")

explainer_lime = LimeTextExplainer(class_names=CLASS_NAMES)
explanation = explainer_lime.explain_instance(
    text_instance=example_ai_spam_text,
    classifier_fn=predictor,
    labels=[example_label_index],
    num_samples=5000,
    top_labels=1
)

print(f"\nLIME Explanation for Class: {CLASS_NAMES[example_label_index]}")
print("Top 10 features contributing to the AI-SPAM classification:")

feature_weights = explanation.as_list(label=example_label_index)

for feature, weight in feature_weights[:10]:
    print(f"  Token: '{feature.replace('\n', ' ')}' | Weight: {weight:.4f}")

# --- 5. Integrated Gradients (IG) Implementation (The SHAP Alternative) ---
print("\n--- 5. Integrated Gradients (IG) Analysis ---")

# 5.1. Define the EMBEDDING attribution function for Captum
def forward_func_embeds(inputs_embeds, attention_mask=None):
    """
    Wrapper function for the model to be compatible with Captum,
    which accepts embeddings and returns logits.
    """
    # inputs_embeds is the tensor Captum will track gradients for
    # We pass the float embeddings directly into the core transformer block (model.electra)
    outputs = model.electra(inputs_embeds=inputs_embeds, attention_mask=attention_mask.long())

    # We take the hidden state (outputs[0]) and pass it to the classification head
    logits = model.classifier(outputs[0])
    return logits

# 5.2. Prepare inputs, embeddings, and baseline (reference)
encoding = tokenizer.encode_plus(
    example_ai_spam_text,
    return_tensors='pt',
    padding='max_length',
    truncation=True,
    max_length=128
)
# We need input_ids to get tokens for display, and attention_mask as an argument
input_ids = encoding['input_ids'].long()
attention_mask = encoding['attention_mask'].long()

# 5.2a. Generate Input Embeddings (The tensor to be attributed)
# We need to detach this from the main graph to use it as a 'fresh' input to ig.attribute
input_embeddings = model.electra.embeddings(input_ids=input_ids).requires_grad_()

# 5.2b. Generate Reference Embeddings (The baseline)
# Use the embedding of the [PAD] token as the baseline
pad_token_id = tokenizer.pad_token_id
pad_embedding = model.electra.embeddings.word_embeddings(torch.tensor(pad_token_id).to(input_ids.device))
reference_embeddings = pad_embedding.unsqueeze(0).expand(input_embeddings.shape)


# 5.3. Initialize and Compute IG
# Initialize IG with the new embedding-based forward function
ig = IntegratedGradients(forward_func_embeds)

print("Calculating Integrated Gradients (Targeting Embeddings)...")

# Compute attribution scores. We attribute the 'input_embeddings' tensor.
attributions_ig, delta = ig.attribute(
    inputs=input_embeddings, # The input tensor for which gradients are computed
    baselines=reference_embeddings, # The baseline tensor
    target=example_label_index,
    n_steps=50,
    internal_batch_size=1,
    additional_forward_args=(attention_mask,), # Mask is a required additional argument
    return_convergence_delta=True
)

# 5.4. Process and Display Numerical Results
# Convert input IDs back to tokens
all_tokens = tokenizer.convert_ids_to_tokens(input_ids.flatten())

# Sum the attributions across the embedding dimension (we only care about the sequence dimension)
# This step is crucial for getting a single score per token
attributions_ig_sum = attributions_ig.sum(dim=-1).squeeze(0)
attributions_ig_sum_numpy = attributions_ig_sum.cpu().detach().numpy()

# Combine tokens and IG scores for text-only output
ig_results = []
for token, score in zip(all_tokens, attributions_ig_sum_numpy):
    # Filter out special tokens
    if token not in ['[CLS]', '[SEP]', '[PAD]'] and not token.startswith('##'):
        ig_results.append((token, score))

# Sort by absolute score to find the most impactful words
ig_results.sort(key=lambda x: abs(x[1]), reverse=True)

print(f"\nIntegrated Gradients Explanation for Class: {CLASS_NAMES[example_label_index]}")
print("Top 10 tokens ranked by absolute IG score (Impact):")
for token, score in ig_results[:10]:
    print(f"  Token: '{token}' | IG Score (Logit): {score:.4f}")

# Optional: Print Convergence Delta to show attribution quality
print(f"\nConvergence Delta (Error Check): {delta.item():.6f}")

# 5.5. Generate and Display Visual Heatmap
print("\nGenerating Visual Heatmap for IG Attributions (HTML output below):")

# Prepare tokens and scores in the format Captum visualization expects
score_list = attributions_ig_sum.tolist()
# Filter out [CLS], [SEP], [PAD] tokens from scores and tokens for cleaner visualization
# Get the indices of non-special tokens
start_index = 1 # Skip [CLS]
end_index = list(input_ids.flatten().cpu().numpy()).index(tokenizer.sep_token_id)
# Extract relevant tokens and scores
visual_tokens = all_tokens[start_index:end_index]
visual_scores = score_list[start_index:end_index]


# Display the visualization using the custom utility (replaces visualize_text)
display(plot_text_heatmap(visual_tokens, visual_scores))

print("\n--- XAI Analysis Complete ---")


--- Initializing Model and Tokenizer ---
Model and Tokenizer loaded successfully.

--- 4. LIME (Local Interpretability) Analysis ---

LIME Explanation for Class: AI-SPAM
Top 10 features contributing to the AI-SPAM classification:
  Token: 'unauthorized' | Weight: -0.0011
  Token: 'confirm' | Weight: -0.0011
  Token: 'and' | Weight: -0.0011
  Token: 'Click' | Weight: -0.0010
  Token: '25' | Weight: -0.0010
  Token: 'identity' | Weight: -0.0009
  Token: 'to' | Weight: -0.0009
  Token: 'Rewards' | Weight: 0.0007
  Token: 'attempt' | Weight: 0.0004
  Token: '2' | Weight: 0.0004

--- 5. Integrated Gradients (IG) Analysis ---
Calculating Integrated Gradients (Targeting Embeddings)...

Integrated Gradients Explanation for Class: AI-SPAM
Top 10 tokens ranked by absolute IG score (Impact):
  Token: '.' | IG Score (Logit): 0.0046
  Token: '.' | IG Score (Logit): 0.0046
  Token: 'been' | IG Score (Logit): 0.0044
  Token: '.' | IG Score (Logit): 0.0041
  Token: 'to' | IG Score (Logit): 0.0039
  To


--- XAI Analysis Complete ---
